In [1]:
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
from scipy.signal import find_peaks


# === Core Functions ===

def create_global_spectrum(X, mz_axis):
    if hasattr(X, "toarray"):  # for sparse matrix
        X = X.toarray()
    global_spectrum = np.sum(X, axis=0)
    return mz_axis, global_spectrum

def detect_peaks(mz_axis, global_spectrum, 
                 min_prominence=0.02, top_n=None,
                 da=0.01,
                 save_path="detected_peaks.txt"):
    
    # Step 1: Peak detection
    peaks, _ = find_peaks(global_spectrum, prominence=min_prominence)
    print(f"Initial detected peaks: {len(peaks)}")

    mz_peaks = mz_axis[peaks]
    intensities = global_spectrum[peaks]

    # Step 2: Sort peaks by intensity descending
    sorted_indices = np.argsort(intensities)[::-1]
    mz_peaks_sorted = mz_peaks[sorted_indices]
    intensities_sorted = intensities[sorted_indices]

    # Step 3: Filter out peaks that are too close (within ±da of a stronger one)
    final_mz = []
    final_intensity = []
    for i, mz in enumerate(mz_peaks_sorted):
        if not any(np.abs(mz - accepted_mz) <= da for accepted_mz in final_mz):
            final_mz.append(mz)
            final_intensity.append(intensities_sorted[i])

    final_mz = np.array(final_mz)
    final_intensity = np.array(final_intensity)

    print(f"Filtered peaks (unique by ±{da} Da): {len(final_mz)}")

    # Step 4: Optional: Top-N filtering after proximity filtering
    if top_n is not None and len(final_mz) > top_n:
        top_indices = np.argsort(final_intensity)[-top_n:][::-1]
        final_mz = final_mz[top_indices]
        final_intensity = final_intensity[top_indices]

    # Step 5: Enhance each final peak by adding intensity of closest neighbor in global spectrum
    for i, mz in enumerate(final_mz):
        mz_diff = np.abs(mz_axis - mz)

        # Ignore the peak itself
        mz_diff[mz_diff == 0] = np.inf

        # Find the closest m/z (excluding the exact match)
        closest_idx = np.argmin(mz_diff)
        if mz_diff[closest_idx] <= da:
            final_intensity[i] += global_spectrum[closest_idx]

    # Step 6: Save to file
    np.savetxt(save_path, np.column_stack((final_mz, final_intensity)),
               fmt="%.6f", header="m/z\tintensity", delimiter="\t")

    return final_mz, final_intensity



In [2]:
# === Parameters ===
input_file = "/Users/amin/Desktop/PhD_Thesis/Code/IMZML Tools/a.h5ad"
#output_file = "/Users/amin/Desktop/PhD_Thesis/Code/IMZML Tools/a_hdi_style_1000_0.04_100.h5ad"

# === Load and Process ===
adata = sc.read_h5ad(input_file)
print(f"Loaded AnnData from: {input_file}")

X = adata.X
X = X.toarray() if hasattr(X, "toarray") else X
row_sums = X.sum(axis=1, keepdims=True)
row_sums[row_sums == 0] = 1  # avoid division by zero
X_norm = X / row_sums

mz_axis, global_spec = create_global_spectrum(X, adata.var_names.astype(float).values)
#mz_axis, global_spec = create_global_spectrum_max(adata.X, adata.var_names.astype(float).values)
peak_mz, peak_intensities = detect_peaks(mz_axis, global_spec, 
                                        min_prominence=0.00000001, top_n=1000,
                                        da=0.02,
                                        save_path="first_top_peaks_00001_002da_add1_glob_1000top_sum.txt")

print(f'Number of peaks: {len(peak_mz)}')


/opt/anaconda3/lib/python3.12/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


Loaded AnnData from: /Users/amin/Desktop/PhD_Thesis/Code/IMZML Tools/a.h5ad
Initial detected peaks: 25194
Filtered peaks (unique by ±0.02 Da): 19747
Number of peaks: 1000
